In [10]:
from comptest.pez import (
    JobArrayIndices,
    LSFJob,
    ResourceRUsage,
    ResourceSelect,
    ResourceSpan,
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from pathlib import Path

job = LSFJob(
    queue="hpc",
    jobname="lossless_projection_tests",
    tasks=16,
    runtime_limit="01:00",
    resources=[
        ResourceSelect("avx512 && model == XeonGold6342"),
        ResourceSpan("hosts=1"),
        ResourceRUsage("mem=2GB"),
    ],
    # array_indices=[1, 2],
    stdout_file=Path("logs/lossless_projection_tests.%J.%I.out").resolve(),
    stderr_file=Path("logs/lossless_projection_tests.%J.%I.err").resolve(),
)

job.to_bsub()

"bsub -q hpc -J lossless_projection_tests -n 16 -W 01:00 -R 'select[avx512 && model == XeonGold6342]' -R 'span[hosts=1]' -R 'rusage[mem=2GB]' -e /zhome/1a/a/219376/Code/qim-naturarv3d-compression/logs/lossless_projection_tests.%J.%I.err -o /zhome/1a/a/219376/Code/qim-naturarv3d-compression/logs/lossless_projection_tests.%J.%I.out"

In [12]:
print(job.to_shell_script(command=["echo", "Hello world!"]))

#!/bin/bash
#BSUB -q hpc
#BSUB -J lossless_projection_tests
#BSUB -n 16
#BSUB -W 01:00
#BSUB -R 'select[avx512 && model == XeonGold6342]'
#BSUB -R 'span[hosts=1]'
#BSUB -R 'rusage[mem=2GB]'
#BSUB -e /zhome/1a/a/219376/Code/qim-naturarv3d-compression/logs/lossless_projection_tests.%J.%I.err
#BSUB -o /zhome/1a/a/219376/Code/qim-naturarv3d-compression/logs/lossless_projection_tests.%J.%I.out

echo 'Hello world!'


In [13]:
# script = job.to_shell_script(command="/bin/echo 'hello world'")
# !bsub <<< "$script"

In [14]:
JobArrayIndices(indices="2:10:2").to_str()

'2-8:2'

In [15]:
# job.submit(command="sleep 10")
from comptest.pez import LSB_JOBINDEX


def func(x, index):
    print("Hello, World!")
    print(f"LSB_INDEX={index}")
    return x**2


print(job.to_shell_script(command=["python3", "-c", "print(5**2)"]))
job.submit_python(func, args=[5], kwargs={"index": LSB_JOBINDEX})

#!/bin/bash
#BSUB -q hpc
#BSUB -J lossless_projection_tests
#BSUB -n 16
#BSUB -W 01:00
#BSUB -R 'select[avx512 && model == XeonGold6342]'
#BSUB -R 'span[hosts=1]'
#BSUB -R 'rusage[mem=2GB]'
#BSUB -e /zhome/1a/a/219376/Code/qim-naturarv3d-compression/logs/lossless_projection_tests.%J.%I.err
#BSUB -o /zhome/1a/a/219376/Code/qim-naturarv3d-compression/logs/lossless_projection_tests.%J.%I.out

python3 -c 'print(5**2)'


Job <28971813> is submitted to queue <hpc>.


In [16]:
job.stdout_file.stem

'lossless_projection_tests.%J.%I'